# Excess Inflation — Swarm Factory

> Generates 81 excess inflation signal variants by sweeping four independent axes, then selects the most differentiated subset for use in a signal swarm. Selection criterion: positive Sharpe + lowest average pairwise return correlation — maximising diversity, not just individual performance.

## Sweep axes
| Axis | Values | Rationale |
|------|--------|-----------|
| **CPI series** | Headline, Core, PCE Core | Different lead/lag: headline spikes first, core confirms |
| **Return window** | 3m, 6m, 12m | Reactivity vs. smoothness |
| **Z-score window** | 36m, 60m, 84m | Regime memory: fast-adapt vs. long-memory |
| **Entry threshold** | 0.5σ, 1.0σ, 1.5σ | Trade frequency vs. conviction |

**3 × 3 × 3 × 3 = 81 variants**

## Selection logic
1. Keep variants with annualised Sharpe > 0
2. Greedy diversification: start with highest-Sharpe variant, iteratively add the one with lowest average correlation to the selected set
3. Export top `N_SELECT` (default 10) as pkl files

## Output
Each selected variant exports `exports/macro_inflation_{variant_id}.pkl` in the standard alpha format `{'long': DataFrame, 'short': DataFrame}` — ready for the td-swarm signal archaeology notebook.

**Run all cells top-to-bottom in a fresh Colab runtime.**
> Requires `FRED_API_KEY` in Colab Secrets.


## Section 0 — Install & Imports

In [ ]:
!pip install fredapi yfinance --quiet
print("Dependencies installed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools, os, pickle, warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

from fredapi import Fred
import yfinance as yf

print('Imports complete')


## Section 1 — Config & Sweep Axes

In [ ]:
from google.colab import userdata
FRED_API_KEY = userdata.get('FRED_API_KEY')

START_DATE = '1990-01-01'
COST_BPS   = 5
N_SELECT   = 10   # number of variants to keep for the swarm

# ── Sweep axes ────────────────────────────────────────────────────────────────
SERIES_MAP = {
    'headline': 'CPIAUCSL',   # headline CPI
    'core':     'CPILFESL',   # core CPI (ex food & energy)
    'pce_core': 'PCEPILFE',   # Fed-preferred: core PCE
}
RETURN_WINDOWS = [3, 6, 12]   # months
Z_WINDOWS      = [36, 60, 84] # months
THRESHOLDS     = [0.5, 1.0, 1.5]

total = len(SERIES_MAP) * len(RETURN_WINDOWS) * len(Z_WINDOWS) * len(THRESHOLDS)
print(f'Sweep: {len(SERIES_MAP)} series × {len(RETURN_WINDOWS)} ret windows × '
      f'{len(Z_WINDOWS)} z-windows × {len(THRESHOLDS)} thresholds = {total} variants')


## Section 2 — Data

In [ ]:
fred = Fred(api_key=FRED_API_KEY)

print('Pulling inflation series from FRED...')
raw_series = {}
for label, series_id in SERIES_MAP.items():
    s = fred.get_series(series_id, observation_start=START_DATE).resample('MS').last()
    raw_series[label] = s
    print(f'  {label:12s} ({series_id}): {s.index[0].date()} → {s.index[-1].date()}')

print('\nDownloading SPY + TLT prices...')
prices_raw = yf.download(['SPY', 'TLT'], start=START_DATE, auto_adjust=True, progress=False)
prices     = prices_raw['Close'].resample('MS').first()
returns    = prices.pct_change()

# Common index across all series + prices
common = prices.index
for s in raw_series.values():
    common = common.intersection(s.index)
prices  = prices.reindex(common)
returns = returns.reindex(common)
for label in raw_series:
    raw_series[label] = raw_series[label].reindex(common)

print(f'\nCommon range : {common[0].date()} → {common[-1].date()} ({len(common)} months)')
print('Data ready')


## Section 3 — Signal & Strategy Functions

In [ ]:
def annualise_pct(series, periods):
    return series.pct_change(periods).apply(lambda x: (1 + x) ** (12 / periods) - 1) * 100

def make_signal(cpi, ret_window, z_window, target=2.0):
    """
    Build excess inflation signal: annualised CPI change - target, z-scored.
    Returns z-score series, lagged 1 month.
    """
    ann  = annualise_pct(cpi, ret_window)
    raw  = (ann - target) / target
    mu   = raw.rolling(z_window, min_periods=z_window // 2).mean()
    sig  = raw.rolling(z_window, min_periods=z_window // 2).std().replace(0, np.nan)
    z    = ((raw - mu) / sig).clip(-3, 3)
    return z.shift(1)   # lag-1: use only data available at trade time

def signal_to_position(signal, threshold):
    """
    Directional: long SPY when signal < -threshold (low inflation = risk-on)
                 long TLT when signal > +threshold (high inflation = defensive)
    Returns SPY position series (0 or 1).
    """
    return pd.Series(
        np.where(signal < -threshold, 1.0, 0.0),
        index=signal.index
    )

def apply_costs(ret_series, position, cost_bps=COST_BPS):
    cost     = cost_bps / 10_000
    pos      = position.reindex(ret_series.index).fillna(0)
    turnover = pos.diff().abs().fillna(0)
    return ret_series * pos - turnover * cost

def perf(r, name=''):
    r = r.dropna()
    if len(r) < 24:
        return pd.Series([np.nan]*5,
            index=['Sharpe','Ann.Ret','Ann.Vol','Max DD','Win Rate'], name=name)
    ann_ret = (1 + r.mean()) ** 12 - 1
    ann_vol = r.std() * np.sqrt(12)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-9 else np.nan
    cum     = (1 + r).cumprod()
    mdd     = (cum / cum.cummax() - 1).min()
    hits    = (r > 0).mean()
    return pd.Series({
        'Sharpe': sharpe, 'Ann.Ret': ann_ret, 'Ann.Vol': ann_vol,
        'Max DD': mdd, 'Win Rate': hits,
    }, name=name)

print('Signal functions defined')


## Section 4 — Generate All 81 Variants

In [ ]:
print(f'Generating {len(SERIES_MAP)*len(RETURN_WINDOWS)*len(Z_WINDOWS)*len(THRESHOLDS)} variants...')

all_returns = {}   # variant_id -> monthly return series
all_signals = {}   # variant_id -> signal series
all_positions = {} # variant_id -> SPY position series

spy_ret = returns['SPY']

for (series_label, ret_w, z_w, thr) in itertools.product(
        SERIES_MAP.keys(), RETURN_WINDOWS, Z_WINDOWS, THRESHOLDS):

    vid = f'{series_label}_r{ret_w}m_z{z_w}m_t{thr}'
    cpi = raw_series[series_label]

    sig = make_signal(cpi, ret_w, z_w)
    pos = signal_to_position(sig, thr)
    ret = apply_costs(spy_ret, pos)

    all_signals[vid]   = sig
    all_positions[vid] = pos
    all_returns[vid]   = ret

print(f'Done. {len(all_returns)} variants generated.')


## Section 5 — Variant Performance Stats

In [ ]:
stats_rows = []
for vid, ret in all_returns.items():
    parts = vid.split('_')
    row   = perf(ret, name=vid).to_dict()
    row['series']    = parts[0]
    row['ret_window'] = int(parts[1][1:-1])
    row['z_window']   = int(parts[2][1:-1])
    row['threshold']  = float(parts[3][1:])
    row['variant_id'] = vid
    stats_rows.append(row)

stats_df = pd.DataFrame(stats_rows).set_index('variant_id')
positive = stats_df[stats_df['Sharpe'] > 0]

print(f'Variants with Sharpe > 0 : {len(positive)} / {len(stats_df)}')
print(f'\nTop 15 by Sharpe:')
print(positive.nlargest(15, 'Sharpe')[['series','ret_window','z_window','threshold','Sharpe','Ann.Ret','Max DD']].round(3).to_string())

# Distribution of Sharpe by axis
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col, title in zip(axes,
        ['series','ret_window','z_window','threshold'],
        ['CPI Series','Return Window (m)','Z-Score Window (m)','Threshold (σ)']):
    stats_df.groupby(col)['Sharpe'].mean().plot(kind='bar', ax=ax, title=f'Avg Sharpe by {title}')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('')
plt.suptitle('Sharpe Distribution Across Sweep Axes', y=1.02)
plt.tight_layout()
plt.show()


## Section 6 — Correlation Analysis

In [ ]:
# Build return matrix for positive-Sharpe variants only
pos_vids = positive.index.tolist()
ret_matrix = pd.DataFrame({vid: all_returns[vid] for vid in pos_vids}).dropna()

corr_matrix = ret_matrix.corr()

# Summary stats
avg_corr = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].mean()
print(f'Average pairwise correlation (positive-Sharpe variants): {avg_corr:.3f}')

# Heatmap — too many to show all; show correlation by series grouping
series_avg_corr = {}
for s1 in SERIES_MAP.keys():
    for s2 in SERIES_MAP.keys():
        v1s = [v for v in pos_vids if v.startswith(s1)]
        v2s = [v for v in pos_vids if v.startswith(s2)]
        if v1s and v2s:
            block = corr_matrix.loc[v1s, v2s].values.flatten()
            series_avg_corr[f'{s1} vs {s2}'] = block.mean()

print('\nAverage cross-series correlation:')
for k, v in series_avg_corr.items():
    print(f'  {k:30s}: {v:.3f}')

# Show correlation among a sample of 20 for visual
sample = pos_vids[:min(20, len(pos_vids))]
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix.loc[sample, sample], vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(sample)))
ax.set_yticks(range(len(sample)))
ax.set_xticklabels([v.replace('_', '\n') for v in sample], fontsize=6, rotation=90)
ax.set_yticklabels([v.replace('_', ' ') for v in sample], fontsize=6)
plt.colorbar(im, ax=ax)
ax.set_title('Pairwise Return Correlation (sample of 20)')
plt.tight_layout()
plt.show()


## Section 7 — Greedy Diversification Selection

In [ ]:
def greedy_diversify(vids, corr_matrix, stats_df, n=N_SELECT):
    """
    Greedy selection: start with highest-Sharpe variant,
    iteratively add the variant with lowest avg correlation to selected set.
    """
    # Sort by Sharpe descending
    ranked = stats_df.loc[vids].sort_values('Sharpe', ascending=False)
    selected = [ranked.index[0]]
    candidates = list(ranked.index[1:])

    while len(selected) < n and candidates:
        # For each candidate, compute avg correlation to already-selected
        avg_corrs = {}
        for cand in candidates:
            c = corr_matrix.loc[selected, cand].mean() if len(selected) > 0 else 0
            avg_corrs[cand] = c
        # Pick candidate with lowest avg correlation to selected set
        best = min(avg_corrs, key=avg_corrs.get)
        selected.append(best)
        candidates.remove(best)

    return selected

selected_vids = greedy_diversify(pos_vids, corr_matrix, positive, n=N_SELECT)

print(f'Selected {len(selected_vids)} variants:')
print()
sel_stats = stats_df.loc[selected_vids, ['series','ret_window','z_window','threshold','Sharpe','Ann.Ret','Max DD']]
print(sel_stats.round(3).to_string())

# Avg pairwise correlation within selected set
sel_corr = corr_matrix.loc[selected_vids, selected_vids]
sel_corr_vals = sel_corr.values[np.triu_indices_from(sel_corr.values, k=1)]
print(f'\nAvg pairwise correlation within selected set: {sel_corr_vals.mean():.3f}')
print(f'(vs {avg_corr:.3f} across all positive-Sharpe variants)')


## Section 8 — Visualise Selected Variants

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1]})

spy_cum = (1 + spy_ret.dropna()).cumprod()
spy_cum.plot(ax=axes[0], color='grey', linestyle='--', linewidth=1.5, label='SPY B&H', alpha=0.8)

cmap = plt.cm.tab10
for i, vid in enumerate(selected_vids):
    r   = all_returns[vid].dropna()
    cum = (1 + r).cumprod()
    cum.plot(ax=axes[0], label=vid.replace('_', ' '), color=cmap(i % 10), linewidth=1)

axes[0].set_title(f'Top {N_SELECT} Selected Inflation Variants — Equity Curves')
axes[0].set_ylabel('Cumulative return')
axes[0].legend(fontsize=7, ncol=2)

# Correlation heatmap of selected
sel_c = corr_matrix.loc[selected_vids, selected_vids]
im = axes[1].imshow(sel_c, vmin=-0.5, vmax=1, cmap='RdBu_r', aspect='auto')
axes[1].set_xticks(range(len(selected_vids)))
axes[1].set_yticks(range(len(selected_vids)))
axes[1].set_xticklabels([v.replace('_', '\n') for v in selected_vids], fontsize=6, rotation=90)
axes[1].set_yticklabels([v.replace('_', ' ') for v in selected_vids], fontsize=6)
plt.colorbar(im, ax=axes[1], label='Correlation')
axes[1].set_title('Pairwise Correlation — Selected Variants')

plt.tight_layout()
plt.show()


## Section 9 — Signal Arrangement & Convoy Leadership

In [ ]:
# Show temporal arrangement: when does each variant enter/exit relative to others?
# Binary active matrix: 1 when in a long position

active_df = pd.DataFrame(
    {vid: all_positions[vid].reindex(returns.index).fillna(0)
     for vid in selected_vids},
    index=returns.index,
)

n_active = active_df.sum(axis=1)

fig, axes = plt.subplots(3, 1, figsize=(14, 9))

# Heatmap of active positions
axes[0].imshow(active_df.T, aspect='auto', cmap='Blues',
               extent=[0, len(active_df), 0, len(selected_vids)])
axes[0].set_yticks(np.arange(len(selected_vids)) + 0.5)
axes[0].set_yticklabels(reversed([v.replace('_',' ') for v in selected_vids]), fontsize=6)
axes[0].set_title('Signal Arrangement — Active Positions Over Time (blue = long SPY)')
axes[0].set_xlabel('Time')

# Number of active variants
n_active.plot(ax=axes[1], color='steelblue', title='Number of Variants Active at Each Bar')
axes[1].axhline(n_active.mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: {n_active.mean():.1f}')
axes[1].legend()

# Entry lead analysis: which variant tends to be first?
# For each "convoy event" (n_active transitions from 0 to >0), record which variant was already active
first_entry_count = {}
for vid in selected_vids:
    pos = active_df[vid]
    # Count how many times this variant was active when n_active first crossed above 1
    convoy_starts = (n_active > 0) & (n_active.shift(1) == 0)
    first_entry_count[vid] = pos[convoy_starts].sum()

fe_series = pd.Series(first_entry_count).sort_values(ascending=True)
fe_series.plot(kind='barh', ax=axes[2],
    title='Convoy Leadership — Times Each Variant Was Active at Start of a New Convoy Event')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.show()

print('Convoy leadership (most frequent first-movers):')
print(fe_series.sort_values(ascending=False).to_string())


## Section 10 — Export PKL Files for Swarm

In [ ]:
os.makedirs('exports', exist_ok=True)

exported = []
for vid in selected_vids:
    pos = all_positions[vid].reindex(returns.index).fillna(0)

    # Standard alpha pkl format: {'long': DataFrame(time x tickers), 'short': DataFrame}
    long_df  = pos.rename('SPY').to_frame()
    short_df = pd.DataFrame(0, index=pos.index, columns=['SPY'])

    path = f'exports/macro_inflation_{vid}.pkl'
    with open(path, 'wb') as f:
        pickle.dump({'long': long_df, 'short': short_df}, f)
    exported.append(path)
    print(f'  Exported  {path}')

print(f'\n{len(exported)} variant signals exported to exports/')
print('Ready for ingestion by td-swarm/notebooks/signal_archaeology.ipynb')
print('\nAdd these to ALPHA_PKL_MAP in the swarm notebook:')
for vid in selected_vids:
    print(f'  \'macro_{vid}\': \'exports/macro_inflation_{vid}.pkl\',')
